In [1]:
import csv
import itertools
import os
from collections import defaultdict

# File paths
BASE_DIR = r"E:\ThegeneBox\problemB_cyp2d6"
ALLELE_DEFS_PATH = os.path.join(BASE_DIR, "allele_definitions.tsv")
VARIANTS_PATH = os.path.join(BASE_DIR, "variants.tsv")
COPY_NUMBER_PATH = os.path.join(BASE_DIR, "copy_number.tsv")
PLATFORM_PATH = os.path.join(BASE_DIR, "platform_content.tsv")
OUT_PATH = os.path.join("out", "problemB_diplotypes.tsv")


def load_tsv(path):
    with open(path, mode="r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f, delimiter="\t"))


def classify_phenotype(as_val, tol=1e-6):
    if abs(as_val - 0.0) <= tol:
        return "Poor"
    elif 0.0 + tol < as_val <= 1.0 + tol:
        return "Intermediate"
    elif 1.0 + tol < as_val <= 2.25 + tol:
        return "Normal"
    elif as_val > 2.25 + tol:
        return "Ultrarapid"
    return "Unknown"


def canonical_diplotype_str(alleles):
    # Order alleles by string length, then lexicographically, joined by ';'
    sorted_alleles = sorted(alleles, key=lambda a: (len(a), a))
    return ";".join(sorted_alleles)


def main():
    allele_defs_rows = load_tsv(ALLELE_DEFS_PATH)
    variants_rows = load_tsv(VARIANTS_PATH)
    cn_rows = load_tsv(COPY_NUMBER_PATH)
    platform_rows = load_tsv(PLATFORM_PATH)

    # 1. Identify assayed variants from platform_content.tsv
    # Adapt column names dynamically if slight variations exist
    var_col_plat = next(c for c in platform_rows[0].keys() if "var" in c.lower() or "id" in c.lower() or "rs" in c.lower())
    assayed_col = next(c for c in platform_rows[0].keys() if "assayed" in c.lower())
    assayed_variants = {
        row[var_col_plat].strip()
        for row in platform_rows
        if row[assayed_col].strip().lower() == "yes"
    }

    # 2. Parse allele definitions & activity values
    allele_col = next(c for c in allele_defs_rows[0].keys() if "allele" in c.lower())
    act_col = next(c for c in allele_defs_rows[0].keys() if "activity" in c.lower())
    var_def_col = next(
        (c for c in allele_defs_rows[0].keys() if "var" in c.lower()), None
    )

    allele_variants = defaultdict(set)
    allele_activity = {}

    for row in allele_defs_rows:
        allele = row[allele_col].strip()
        allele_activity[allele] = float(row[act_col].strip())
        if var_def_col and row[var_def_col].strip() not in ("", "NA", "None", "."):
            # Handle comma/semicolon-separated variant lists per row or one-variant-per-row
            raw_vars = row[var_def_col].replace(";", ",").split(",")
            for v in raw_vars:
                v_clean = v.strip()
                if v_clean:
                    allele_variants[allele].add(v_clean)

    # Ensure *5 is registered if present in definitions, excluded from non-deletion pool
    star5_activity = allele_activity.get("*5", 0.0)
    non_deletion_alleles = sorted([a for a in allele_activity.keys() if a != "*5"])

    # 3. Parse copy_number.tsv
    sample_col_cn = next(c for c in cn_rows[0].keys() if "sample" in c.lower())
    cn_col = next(c for c in cn_rows[0].keys() if "cn" in c.lower() or "copy" in c.lower())
    sample_cn_raw = {}
    for row in cn_rows:
        sample_cn_raw[row[sample_col_cn].strip()] = row[cn_col].strip()

    # 4. Parse variants.tsv (supports both long format and wide matrix format)
    var_headers = list(variants_rows[0].keys())
    sample_genotypes = defaultdict(dict)

    if any("sample" in h.lower() for h in var_headers) and any("gt" in h.lower() or "geno" in h.lower() for h in var_headers):
        # Long format: sample_id, variant_id, genotype
        s_col = next(h for h in var_headers if "sample" in h.lower())
        v_col = next(h for h in var_headers if "var" in h.lower() or "rs" in h.lower())
        g_col = next(h for h in var_headers if "gt" in h.lower() or "geno" in h.lower())
        for row in variants_rows:
            sample_genotypes[row[s_col].strip()][row[v_col].strip()] = row[g_col].strip()
    else:
        # Wide format: first column is variant_id, remaining columns are sample_ids
        v_col = var_headers[0]
        sample_cols = var_headers[1:]
        for row in variants_rows:
            v_id = row[v_col].strip()
            for s_id in sample_cols:
                sample_genotypes[s_id.strip()][v_id] = row[s_id].strip()

    # 5. Evaluate each sample
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    out_fields = [
        "sample_id",
        "diplotypes",
        "activity_score_min",
        "activity_score_max",
        "phenotypes",
        "ambiguous",
        "reason_code",
    ]

    with open(OUT_PATH, mode="w", encoding="utf-8", newline="") as out_f:
        writer = csv.DictWriter(out_f, fieldnames=out_fields, delimiter="\t")
        writer.writeheader()

        for sample_id, raw_cn in sample_cn_raw.items():
            cn_is_na = raw_cn.upper() == "NA" or raw_cn == ""
            total_cn = 2 if cn_is_na else int(float(raw_cn))
            obs_gts = sample_genotypes.get(sample_id, {})

            valid_solutions = []

            # Enumerate all multisets of size total_cn from non-deletion alleles
            for combo in itertools.combinations_with_replacement(non_deletion_alleles, total_cn):
                consistent = True
                for var_id in assayed_variants:
                    gt = obs_gts.get(var_id, ".")
                    if gt == ".":
                        continue

                    # Count k = number of non-deletion copies carrying var_id
                    k = sum(1 for a in combo if var_id in allele_variants[a])

                    if gt == "0/0" and k != 0:
                        consistent = False
                        break
                    elif gt == "1/1" and k != total_cn:
                        consistent = False
                        break
                    elif gt == "0/1" and not (1 <= k <= total_cn - 1):
                        consistent = False
                        break

                if consistent:
                    full_multiset = list(combo)
                    if total_cn < 2:
                        full_multiset.extend(["*5"] * (2 - total_cn))
                    valid_solutions.append(full_multiset)

            # Determine reason code by priority
            num_sols = len(valid_solutions)
            if num_sols == 0:
                writer.writerow({
                    "sample_id": sample_id,
                    "diplotypes": "NA",
                    "activity_score_min": "NA",
                    "activity_score_max": "NA",
                    "phenotypes": "NA",
                    "ambiguous": "NA",
                    "reason_code": "NO_CONSISTENT_DIPLOTYPE",
                })
                continue

            if cn_is_na:
                reason_code = "CN_UNKNOWN"
            elif num_sols == 1:
                reason_code = "RESOLVED"
            elif total_cn > 2:
                reason_code = "AMBIG_CN_ASSIGNMENT"
            else:
                reason_code = "AMBIG_UNASSAYED_VARIANT"

            # Format diplotypes, activity scores, and phenotypes
            diplotype_strs = [canonical_diplotype_str(sol) for sol in valid_solutions]
            scores = [
                sum(star5_activity if a == "*5" else allele_activity[a] for a in sol)
                for sol in valid_solutions
            ]

            min_as = min(scores)
            max_as = max(scores)

            # Deduplicate phenotypes while preserving order of appearance
            seen_phenos = []
            for sc in scores:
                p = classify_phenotype(sc)
                if p not in seen_phenos:
                    seen_phenos.append(p)

            writer.writerow({
                "sample_id": sample_id,
                "diplotypes": "|".join(diplotype_strs),
                "activity_score_min": f"{min_as:g}",
                "activity_score_max": f"{max_as:g}",
                "phenotypes": "|".join(seen_phenos),
                "ambiguous": "yes" if num_sols > 1 else "no",
                "reason_code": reason_code,
            })

    print(f"Done. Wrote results to {OUT_PATH}")


if __name__ == "__main__":
    main()

Done. Wrote results to out\problemB_diplotypes.tsv


In [2]:
import csv
import itertools
import os
from collections import defaultdict

# File paths
BASE_DIR = r"E:\ThegeneBox\problemB_cyp2d6"
ALLELE_DEFS_PATH = os.path.join(BASE_DIR, "allele_definitions.tsv")
VARIANTS_PATH = os.path.join(BASE_DIR, "variants.tsv")
COPY_NUMBER_PATH = os.path.join(BASE_DIR, "copy_number.tsv")
PLATFORM_PATH = os.path.join(BASE_DIR, "platform_content.tsv")
OUT_PATH = r"E:\ThegeneBox\out\problemB_diplotypes.tsv"


def load_tsv(path):
    with open(path, mode="r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f, delimiter="\t"))


def classify_phenotype(as_val, tol=1e-6):
    if abs(as_val - 0.0) <= tol:
        return "Poor"
    elif 0.0 + tol < as_val <= 1.0 + tol:
        return "Intermediate"
    elif 1.0 + tol < as_val <= 2.25 + tol:
        return "Normal"
    elif as_val > 2.25 + tol:
        return "Ultrarapid"
    return "Unknown"


def canonical_diplotype_str(alleles):
    # Order alleles by string length, then lexicographically, joined by ';'
    sorted_alleles = sorted(alleles, key=lambda a: (len(a), a))
    return ";".join(sorted_alleles)


def main():
    # Step 1: Load input TSV files
    allele_defs_rows = load_tsv(ALLELE_DEFS_PATH)
    variants_rows = load_tsv(VARIANTS_PATH)
    cn_rows = load_tsv(COPY_NUMBER_PATH)
    platform_rows = load_tsv(PLATFORM_PATH)
    print("step 1 completed: Loaded all 4 input TSV files.")

    # Step 2: Filter assayed variants from platform_content.tsv
    assayed_variants = {
        row["variant_id"].strip()
        for row in platform_rows
        if row["assayed_on_platform"].strip().lower() == "yes"
    }
    print("step 2 completed: Filtered assayed variants on platform.")

    # Step 3: Parse allele definitions, activity values, and non-deletion pool
    allele_variants = {}
    allele_activity = {}
    star5_activity = 0.0

    for row in allele_defs_rows:
        allele = row["allele"].strip()
        act_val = float(row["activity_value"].strip())
        is_del = row["is_deletion"].strip().lower() == "yes"
        raw_vars = row["defining_variants"].strip()

        allele_activity[allele] = act_val
        if is_del or allele == "*5":
            star5_activity = act_val
        else:
            if raw_vars in ("-", "", "NA"):
                allele_variants[allele] = set()
            else:
                allele_variants[allele] = {
                    v.strip() for v in raw_vars.split(",") if v.strip()
                }

    non_deletion_alleles = sorted(allele_variants.keys())
    print("step 3 completed: Parsed allele definitions and separated deletion (*5) from non-deletion alleles.")

    # Step 4: Parse copy number and long-format variant genotypes
    sample_cn_list = [
        (row["sample_id"].strip(), row["total_cn"].strip()) for row in cn_rows
    ]

    var_headers = list(variants_rows[0].keys())
    gt_col = var_headers[2]
    sample_genotypes = defaultdict(dict)

    for row in variants_rows:
        s_id = row["sample_id"].strip()
        v_id = row["variant_id"].strip()
        gt_val = row[gt_col].strip() if row[gt_col] is not None else "."
        sample_genotypes[s_id][v_id] = gt_val if gt_val != "" else "."
    print("step 4 completed: Parsed copy numbers and sample variant genotypes.")

    # Step 5: Enumerate consistent multisets, calculate activity scores, and write output
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    out_fields = [
        "sample_id",
        "diplotypes",
        "activity_score_min",
        "activity_score_max",
        "phenotypes",
        "ambiguous",
        "reason_code",
    ]

    with open(OUT_PATH, mode="w", encoding="utf-8", newline="") as out_f:
        writer = csv.DictWriter(out_f, fieldnames=out_fields, delimiter="\t")
        writer.writeheader()

        for sample_id, raw_cn in sample_cn_list:
            cn_is_na = raw_cn.upper() == "NA" or raw_cn == ""
            total_cn = 2 if cn_is_na else int(float(raw_cn))
            obs_gts = sample_genotypes.get(sample_id, {})

            valid_solutions = []

            for combo in itertools.combinations_with_replacement(
                non_deletion_alleles, total_cn
            ):
                consistent = True
                for var_id in assayed_variants:
                    gt = obs_gts.get(var_id, ".")
                    if gt == ".":
                        continue

                    k = sum(1 for a in combo if var_id in allele_variants[a])

                    if gt == "0/0" and k != 0:
                        consistent = False
                        break
                    elif gt == "1/1" and k != total_cn:
                        consistent = False
                        break
                    elif gt == "0/1" and not (1 <= k <= total_cn - 1):
                        consistent = False
                        break

                if consistent:
                    full_multiset = list(combo)
                    if total_cn < 2:
                        full_multiset.extend(["*5"] * (2 - total_cn))
                    valid_solutions.append(full_multiset)

            num_sols = len(valid_solutions)
            if num_sols == 0:
                writer.writerow(
                    {
                        "sample_id": sample_id,
                        "diplotypes": "NA",
                        "activity_score_min": "NA",
                        "activity_score_max": "NA",
                        "phenotypes": "NA",
                        "ambiguous": "NA",
                        "reason_code": "NO_CONSISTENT_DIPLOTYPE",
                    }
                )
                continue

            if cn_is_na:
                reason_code = "CN_UNKNOWN"
            elif num_sols == 1:
                reason_code = "RESOLVED"
            elif total_cn > 2:
                reason_code = "AMBIG_CN_ASSIGNMENT"
            else:
                reason_code = "AMBIG_UNASSAYED_VARIANT"

            diplotype_strs = [canonical_diplotype_str(sol) for sol in valid_solutions]
            scores = [
                sum(star5_activity if a == "*5" else allele_activity[a] for a in sol)
                for sol in valid_solutions
            ]

            min_as = min(scores)
            max_as = max(scores)

            seen_phenos = []
            for sc in scores:
                p = classify_phenotype(sc)
                if p not in seen_phenos:
                    seen_phenos.append(p)

            writer.writerow(
                {
                    "sample_id": sample_id,
                    "diplotypes": "|".join(diplotype_strs),
                    "activity_score_min": f"{min_as:g}",
                    "activity_score_max": f"{max_as:g}",
                    "phenotypes": "|".join(seen_phenos),
                    "ambiguous": "yes" if num_sols > 1 else "no",
                    "reason_code": reason_code,
                }
            )

    print(f"step 5 completed: Diplotype calling finished and saved to {OUT_PATH}.")


if __name__ == "__main__":
    main()

step 1 completed: Loaded all 4 input TSV files.
step 2 completed: Filtered assayed variants on platform.
step 3 completed: Parsed allele definitions and separated deletion (*5) from non-deletion alleles.
step 4 completed: Parsed copy numbers and sample variant genotypes.
step 5 completed: Diplotype calling finished and saved to E:\ThegeneBox\out\problemB_diplotypes.tsv.
